# Day 16: Graphs - Shortest Path

**Week 3 — Advanced DSA**

---

## 📚 Theory

Yesterday we used DFS/BFS to explore connected components. Today, we focus on finding the **Shortest Path**.

### BFS (Unweighted Graphs)
For graphs where every edge has the same weight (or no weight), BFS is guaranteed to find the shortest path. 
- **Multi-Source BFS**: Sometimes you have multiple starting points (e.g., "zombies" spreading). Add ALL starting points to the queue at time `t=0`.

### Dijkstra's Algorithm (Weighted Graphs)
When edges have different positive weights, a simple queue won't work. We must use a **Priority Queue (Min-Heap)**.
- Instead of a standard queue storing `(node)`, store `(current_distance, node)`.
- Always pop the node with the smallest distance.
- Runs in `O(E log V)` time where E is edges and V is vertices.

### Prim's Algorithm (Minimum Spanning Tree)
Used to connect all nodes with the minimum possible total edge weight.
- Very similar to Dijkstra's, but the Min-Heap stores `(edge_cost, node)` and we don't accumulate the total path distance—we just care about the cheapest next edge.

## 🔨 Practice Problem 1: Rotting Oranges (Medium)

**LeetCode #994** | [Link](https://leetcode.com/problems/rotting-oranges/)

**Why it's relevant**: The absolute best problem to learn Multi-Source BFS on a grid.

**Problem**: You are given an `m x n` `grid` where each cell can have one of three values:
- `0` representing an empty cell,
- `1` representing a fresh orange, or
- `2` representing a rotten orange.
Every minute, any fresh orange that is 4-directionally adjacent to a rotten orange becomes rotten. Return the minimum number of minutes that must elapse until no cell has a fresh orange. If this is impossible, return `-1`.

**Approach**:
1. Scan the grid. Count all fresh oranges. Add the coordinates of ALL rotten oranges to a `queue`.
2. Run BFS level-by-level (using `for _ in range(len(queue))`). Every level represents 1 minute.
3. Squeeze fresh oranges into rotten ones, decrementing your fresh count.

In [ ]:
import collections

def oranges_rotting(grid: list[list[int]]) -> int:
    """Return the minimum minutes for all oranges to rot."""
    # YOUR SOLUTION HERE
    pass

# Test cases
assert oranges_rotting([[2,1,1],[1,1,0],[0,1,1]]) == 4
assert oranges_rotting([[2,1,1],[0,1,1],[1,0,1]]) == -1
assert oranges_rotting([[0,2]]) == 0
print("All tests passed! ✅")

In [ ]:
# Solution: Rotting Oranges
def oranges_rotting_solution(grid: list[list[int]]) -> int:
    """O(M*N) time, O(M*N) space."""
    rows, cols = len(grid), len(grid[0])
    queue = collections.deque()
    fresh, time = 0, 0
    
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                fresh += 1
            elif grid[r][c] == 2:
                queue.append([r, c])
                
    directions = [[0,1], [0,-1], [1,0], [-1,0]]
    
    while queue and fresh > 0:
        # Process an entire level (1 minute of time)
        for i in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in directions:
                row, col = r + dr, c + dc
                # If in bounds and fresh, make it rotten
                if (0 <= row < rows and 
                    0 <= col < cols and 
                    grid[row][col] == 1):
                    grid[row][col] = 2
                    queue.append([row, col])
                    fresh -= 1
        time += 1
        
    return time if fresh == 0 else -1


## 🔨 Practice Problem 2: Network Delay Time (Medium)

**LeetCode #743** | [Link](https://leetcode.com/problems/network-delay-time/)

**Why it's relevant**: The canonical implementation of Dijkstra's Algorithm.

**Problem**: You are given a network of `n` nodes, labeled from `1` to `n`. You are also given `times`, a list of travel times as directed edges `times[i] = (ui, vi, wi)`, where `wi` is the time it takes for a signal to travel from node `ui` to node `vi`.
We send a signal from a given node `k`. Return the minimum time it takes for all the `n` nodes to receive the signal. If it is impossible, return `-1`.

**Approach**:
1. Build an adjacency list: `adj = {node: [(weight, neighbor)]}`.
2. Use a Min-Heap initialized with `(0, k)` representing `(total_time, node)`.
3. Keep a `visit` set. Pop from heap, add to `visit`, record the max time, and push all neighbors with `(current_time + neighbor_weight)` to the heap.

In [ ]:
import heapq

def network_delay_time(times: list[list[int]], n: int, k: int) -> int:
    """Return min time for all nodes to receive signal."""
    # YOUR SOLUTION HERE
    pass

# Test cases
assert network_delay_time([[2,1,1],[2,3,1],[3,4,1]], 4, 2) == 2
assert network_delay_time([[1,2,1]], 2, 1) == 1
assert network_delay_time([[1,2,1]], 2, 2) == -1
print("All tests passed! ✅")

In [ ]:
# Solution: Network Delay Time
def network_delay_time_solution(times: list[list[int]], n: int, k: int) -> int:
    """O(E log V) time, O(V + E) space."""
    edges = collections.defaultdict(list)
    for u, v, w in times:
        edges[u].append((w, v))
        
    min_heap = [(0, k)]
    visit = set()
    t = 0
    
    while min_heap:
        w1, n1 = heapq.heappop(min_heap)
        if n1 in visit:
            continue
            
        visit.add(n1)
        t = w1  # Because we use a min_heap, the time naturally increases monotonically
        
        for w2, n2 in edges[n1]:
            if n2 not in visit:
                heapq.heappush(min_heap, (w1 + w2, n2))
                
    return t if len(visit) == n else -1


## 🔨 Practice Problem 3: Min Cost to Connect All Points (Medium)

**LeetCode #1584** | [Link](https://leetcode.com/problems/min-cost-to-connect-all-points/)

**Why it's relevant**: The canonical implementation of Prim's Algorithm for Minimum Spanning Trees (MST).

**Problem**: You are given an array `points` representing integer coordinates of some points on a 2D-plane. The cost of connecting two points `[xi, yi]` and `[xj, yj]` is the manhattan distance between them: `|xi - xj| + |yi - yj|`.
Return the minimum cost to make all points connected (i.e. to form a Minimum Spanning Tree).

**Approach**:
Like Dijkstra's, use a Min-Heap. Start at point `0`, add it to a `visited` set. Calculate distance to all unvisited nodes, push `(cost, neighbor)` to heap. Pop the cheapest edge, add node to `visited`, add cost to total, repeat!

In [ ]:
def min_cost_connect_points(points: list[list[int]]) -> int:
    """Return the minimum cost to connect all points."""
    # YOUR SOLUTION HERE
    pass

# Test cases
assert min_cost_connect_points([[0,0],[2,2],[3,10],[5,2],[7,0]]) == 20
assert min_cost_connect_points([[3,12],[-2,5],[-4,1]]) == 18
print("All tests passed! ✅")

In [ ]:
# Solution: Min Cost to Connect All Points
def min_cost_connect_points_solution(points: list[list[int]]) -> int:
    """O(N^2 log N) time, O(N^2) space."""
    N = len(points)
    
    # Build adjacency list: adj[i] = [(cost, j)]
    adj = {i: [] for i in range(N)}
    for i in range(N):
        x1, y1 = points[i]
        for j in range(i + 1, N):
            x2, y2 = points[j]
            dist = abs(x1 - x2) + abs(y1 - y2)
            adj[i].append([dist, j])
            adj[j].append([dist, i])
            
    # Prim's Algorithm
    res = 0
    visit = set()
    min_heap = [[0, 0]]  # [cost, node]
    
    while len(visit) < N:
        cost, i = heapq.heappop(min_heap)
        if i in visit:
            continue
            
        res += cost
        visit.add(i)
        
        for neighbor_cost, neighbor_idx in adj[i]:
            if neighbor_idx not in visit:
                heapq.heappush(min_heap, [neighbor_cost, neighbor_idx])
                
    return res


## 🏆 Daily Challenge

Without looking at solutions, attempt this final Google-classic challenge:

**Word Ladder (Hard) - LeetCode #127**

**Problem**: A transformation sequence from word `beginWord` to word `endWord` using a dictionary `wordList` is a sequence of words such that:
- Every adjacent pair of words differs by a single letter.
- Every word in the sequence is in `wordList`.

Return the number of words in the shortest transformation sequence from `beginWord` to `endWord`, or 0 if no such sequence exists.

**Hint**: This is an implicitly unweighted graph. The shortest path on an unweighted graph is just BFS! The tricky part is generating the "neighbors" (words that differ by 1 character). Use a wildcard hashmap `{*ot: [hot, dot, lot]}` to generate adjacency quickly.

In [ ]:
def ladder_length(beginWord: str, endWord: str, wordList: list[str]) -> int:
    """Return shortest sequence length."""
    # YOUR SOLUTION HERE
    pass

# Test cases
assert ladder_length("hit", "cog", ["hot","dot","dog","lot","log","cog"]) == 5
assert ladder_length("hit", "cog", ["hot","dot","dog","lot","log"]) == 0
print("Daily challenge complete! 🏆")

In [ ]:
# Solution: Word Ladder
def ladder_length_solution(beginWord: str, endWord: str, wordList: list[str]) -> int:
    """O(N * M^2) time where N is words and M is length of word."""
    if endWord not in wordList:
        return 0
        
    # Add beginWord to list to build map easily
    wordList.append(beginWord)
    
    # Build adjacency list using patterns
    # pattern -> list of words matching that pattern
    nei = collections.defaultdict(list)
    for word in wordList:
        for j in range(len(word)):
            pattern = word[:j] + "*" + word[j+1:]
            nei[pattern].append(word)
            
    visit = set([beginWord])
    queue = collections.deque([beginWord])
    res = 1
    
    while queue:
        for i in range(len(queue)):
            word = queue.popleft()
            if word == endWord:
                return res
                
            # Find all neighbors
            for j in range(len(word)):
                pattern = word[:j] + "*" + word[j+1:]
                for neighbor in nei[pattern]:
                    if neighbor not in visit:
                        visit.add(neighbor)
                        queue.append(neighbor)
        res += 1
        
    return 0


## 📝 Day 16 Review

### Patterns Learned Today
- [ ] Multi-Source BFS on Grids by seeding the initial queue with multiple elements (`Rotting Oranges`).
- [ ] Implementing **Dijkstra's Algorithm** with a Min-Heap to find shortest paths on weighted graphs (`Network Delay Time`).
- [ ] Implementing **Prim's Algorithm** to construct Minimum Spanning Trees (`Min Cost to Connect All Points`).
- [ ] Using abstract string transformations to model unweighted edges for BFS (`Word Ladder`).

### Tomorrow's Preview
**Day 17: Dynamic Programming 1D** — We enter the final boss of DSA. We will learn how to transition from recursive backtracking to memoization to tabulated DP.